In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_csv(r"C:\Users\shree\Downloads\train.csv\train.csv")

In [7]:
df.shape

(404290, 6)

In [9]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [13]:
new_df = df.sample(30000) # as the data is so large we are working on a subset for ease

In [15]:
new_df.isnull().sum()

id              0
qid1            0
qid2            0
question1       0
question2       0
is_duplicate    0
dtype: int64

In [17]:
new_df.duplicated().sum()

0

In [19]:
ques_df = new_df[['question1','question2']]
ques_df.head()

,question1,question2
265300,I'm 17 year old my height is 5 feet 4 inch and...,I'm am 15 years old. I weigh 55 kg and my heig...
26870,What could be causing my iPhone 5 to keep rebo...,Why does my iPhone 5 keep showing no service a...
404090,What is the main quality you think makes a goo...,I ordered an power bank from snapdeal.Yesterda...
309384,Why does the Greek language sound like Spanish?,Why does the Greek language sound so much like...
360139,What are some fun things to do at night with f...,What are some fun things to do at a sleepover?


In [21]:
from sklearn.feature_extraction.text import CountVectorizer
# merge texts
questions = list(ques_df['question1']) + list(ques_df['question2'])

cv = CountVectorizer(max_features=3000) # Create BoW vectorizer; keep at most 3000 vocabulary features
q1_arr, q2_arr = np.vsplit(cv.fit_transform(questions).toarray(),2)
# fit = learn vocabulary
# transform = convert text → numerical vectors
# .toarray() Convert sparse matrix → normal NumPy array
# np.vsplit(..., 2) Split rows into 2 equal parts

# q1_arr, q2_arr = ...
# First half = question1 vectors
# Second half = question2 vectors

In [23]:
temp_df1 = pd.DataFrame(q1_arr, index= ques_df.index)
temp_df2 = pd.DataFrame(q2_arr, index= ques_df.index)
temp_df = pd.concat([temp_df1, temp_df2], axis=1)
temp_df.shape

(30000, 6000)

In [25]:
temp_df

,0,1,2,3,4,5,6,7,8,9,...,2990,2991,2992,2993,2994,2995,2996,2997,2998,2999
265300,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
26870,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
404090,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
309384,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
360139,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296113,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
354658,0,0,0,1,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
303306,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
230676,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
temp_df['is_duplicate'] = new_df['is_duplicate']

In [29]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(temp_df.iloc[:,0:-1].values,temp_df.iloc[:,-1].values,test_size=0.2,random_state=1)

In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.747

In [33]:
from xgboost import XGBClassifier
xgb = XGBClassifier()
xgb.fit(X_train,y_train)
y_pred = xgb.predict(X_test)
accuracy_score(y_test,y_pred)

ModuleNotFoundError: No module named 'xgboost'